# Milestone 1 - Basic AI Model Training


## Environment Setup & Hardware Verification

Hardware detection and package installation for running locally or in Colab.

In [ ]:
# System and hardware verification - putting this in due to the numerous issues I have had on my local machine with this.
import sys
import platform
import os

print(f"Python version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")

try:
    import torch
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")
    if cuda_available:
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not installed; skipping GPU detection.")

In [ ]:
%pip install -q --upgrade pip
%pip install -q pandas numpy scipy matplotlib seaborn scikit-learn pyarrow fastparquet openpyxl google-cloud-storage

In [ ]:
# Core dependencies verification
try:
    import numpy as np
    import pandas as pd
    import scipy
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    import sklearn
    import pyarrow
    import openpyxl
    print("Core libraries imported successfully.")
except ImportError as e:
    print(f"Libraries not yet installed in current runtime ({e}).")
    print("Run the %pip install cell above when running in Google Colab.")

In [ ]:
from pathlib import Path

# Project root directory setup
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

for p in [DATA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"Directory ready: {p.relative_to(BASE_DIR)}")

## 1. Raw Data Storage

The raw data for this project is `Attack_Dataset.csv`. This csv file (in this repo) will be copied into the notebook and then into the bucket.

#### How I came up with this approach:

- **ephemeral nature of colab runtimes**: Every time you exit your Colab session (or let it go idle), the runtime gets destroyed. The same applies to its local disk space. If all your raw files were stored solely on the Colab instance's disk, they'd get lost after each run. By storing the base file in a GCP bucket, I'm making sure it persists through multiple runs of the notebook.
- **immutability of the raw data**: Per the lectures, raw data should never change. A raw data file could be useful in case a problem comes to light later when cleaning/preprocessing features in a subsequent step; if the raw file has been preserved from the beginning, I'll always have something I can use to rerun everything from a clean slate without worrying about corrupting the data. In my code, I set the local cached version to be read only so I can't accidentally changes it.
- **keeping data out of git** : It is generally considered bad practice to include raw data directly in the Git repository as was shown in lecture 2. The Git repository is meant to track versions of code/configurations/checksums, whereas the raw data goes into the storage system.
- **avoid I/O overhead** : With Lecture 2 in mind, reading lines from the network every time I train an epoch/model or preprocess some subset of a larger dataset can lead to huge I/O overhead and leaves my compute resources unused. By caching `Attack_Dataset.csv` locally to the NVMe SSD (once) at `/data/raw`, we enable fast sequential access for both Pandas and PyTorch.

In [ ]:
from pathlib import Path
import os
import stat
import shutil

# Storage configuration
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_FILE = RAW_DATA_DIR / "Attack_Dataset.csv"

# Locate Attack_Dataset.csv in repo or local workspace and stage to RAW_DATA_FILE
if not RAW_DATA_FILE.exists():
    possible_paths = [
        BASE_DIR / "Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv",
        BASE_DIR / "Attack_Dataset.csv",
        Path("Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv"),
        Path("Attack_Dataset.csv"),
        Path("../Attack_Dataset.csv"),
    ]
    for p in possible_paths:
        if p.exists():
            shutil.copy(p, RAW_DATA_FILE)
            print(f"Staged dataset from {p} to {RAW_DATA_FILE}")
            break

GCS_PROJECT_ID = os.environ.get("GCP_PROJECT", "setu-ai-hdip")
GCS_BUCKET_NAME = "setu-hdip-ai-raw"
GCS_REGION = "europe-west1"
GCS_BLOB_PATH = "cybersecurity/Attack_Dataset.csv"
REMOTE_GCS_URI = f"gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}"

print(f"Target GCS Bucket: gs://{GCS_BUCKET_NAME}")
print(f"Local Raw Path:    {RAW_DATA_FILE}")

# Spin up GCS databucket and sync raw file
# (uncomment auth in Colab if authenticating for the first time)
# from google.colab import auth
# auth.authenticate_user()

try:
    from google.cloud import storage
    storage_client = storage.Client(project=GCS_PROJECT_ID)
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    
    if not bucket.exists():
        print(f"Spinning up GCS bucket: gs://{GCS_BUCKET_NAME} ({GCS_REGION})...")
        bucket = storage_client.create_bucket(GCS_BUCKET_NAME, location=GCS_REGION)
        print(f"Bucket ready: gs://{GCS_BUCKET_NAME}")
    else:
        print(f"Connected to existing bucket: gs://{GCS_BUCKET_NAME}")
        
    # Sync to GCS if local file is present
    if RAW_DATA_FILE.exists():
        blob = bucket.blob(GCS_BLOB_PATH)
        if not blob.exists():
            print(f"Uploading {RAW_DATA_FILE.name} to gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}...")
            blob.upload_from_filename(str(RAW_DATA_FILE))
            print("Upload to bucket complete.")
        else:
            print(f"Blob already exists in bucket: gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}")
except ImportError:
    print("Note: google-cloud-storage not installed in current environment.")
except Exception as e:
    print(f"GCS bucket status note: {e}")

# Enforce read-only immutability on local raw file
if RAW_DATA_FILE.exists():
    try:
        os.chmod(RAW_DATA_FILE, stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)
        print("Permissions: Set to read-only (immutable).")
    except Exception as e:
        print(f"Note on permissions: {e}")
else:
    print(f"Local raw file '{RAW_DATA_FILE.name}' not found yet. It can be uploaded or synced in Section 5 (Data Access).")

## 2. Processed Data Storage & File Formats

The location of the processed feature tables and cleaned datasets will follow a two-tier structure based on the source raw data:

- **persistent cloud-based storage object**: `gs://setu-hdip-ai-raw/cybersecurity/processed/` stores my data in gcp.
- **fast training access local working cache**: `./data/processed/` on the ssd of the local Colab instance.
- **raw format (CSV)**: the `Attack_Dataset.csv` is in CSV format as this is the original format that was retrieved from the public dataset. CSV is portable and human readable, however, it is extremely inefficient for model training where features need to be iteratively loaded.
- **processed format (Apache Parquet)**: the cleaned features and their respective encoding/scale transformations will be stored in `attack_features_v1.0.parquet`, utilizing the `pyarrow` engine with snappy compression. (note to self - is parquet needed? must check, i may be misunderstanding it and its functionality)
- **columnar storage (ref slide 21 lecture 2)**: Unlike CSV that stores data row-by-row, Apache Parquet stores data column-by-column. Since most ML feature engineering workflows involve selecting subsets of columns (i.e., numerical vs categorical encoded features), columnar storage means you avoid loading unused columns into RAM.

In [ ]:
from pathlib import Path
import os

# 1. Directory and file paths for processed data
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PARQUET_FILE = PROCESSED_DATA_DIR / "attack_features_v1.0.parquet"
REMOTE_PROCESSED_URI = "gs://setu-hdip-ai-raw/cybersecurity/processed/attack_features_v1.0.parquet"

print(f"Local processed directory: {PROCESSED_DATA_DIR.relative_to(BASE_DIR)}")
print(f"Local Parquet target:      {PROCESSED_PARQUET_FILE}")
print(f"Remote persistent URI:     {REMOTE_PROCESSED_URI}")

# 2. Helper function to export processed data to Parquet
def export_to_parquet(df, target_path: Path) -> Path:
    """
    Exports processed DataFrame to Apache Parquet format using pyarrow
    and Snappy compression, then prints file size.
    """
    # Clean trailing empty unnamed column if present
    cols_to_save = [c for c in df.columns if not str(c).startswith("Unnamed")]
    df[cols_to_save].to_parquet(target_path, engine="pyarrow", compression="snappy", index=False)
    size_kb = target_path.stat().st_size / 1024
    print(f"Exported Parquet file: {target_path.name} ({size_kb:.2f} KB)")
    return target_path

# Preview conversion if raw data is already cached
if RAW_DATA_FILE.exists():
    try:
        import pandas as pd
        sample_df = pd.read_csv(RAW_DATA_FILE, nrows=50)
        export_to_parquet(sample_df, PROCESSED_PARQUET_FILE)
        print("Sample Parquet conversion verified successfully.")
    except Exception as e:
        print(f"Note on sample export: {e}")
else:
    print("Ready for processed feature exports once raw data is staged.")

## 3. Database / Object Storage Decision

(ref lecture 3 slide 33) i assessed to determine which type of storage architecture was best suited to support this project’s workflow; a Relational Database or object storage with local file system files.

### relational databases (sql - postgresql/mysql)
Relational databases provide the most support for OLTP workloads (i.e., online transaction processing), provide strong consistency guarantees, allow for multiple users to lock rows simultaneously (CRDT like quality) and enable complex join operations (ref Lecture 2 slide 22). Although `Attack_Dataset.csv` contains data that can be organized into a table format, all aspects of the model training pipeline used by this project will involve offline batch training. Each iteration of the training loop reads through the input data sequentially in mini batches (thus there are no need for random row lookups (`where id=x`) as this wouldnt be suitable for this projects needs and goals). Therefore, the addition of an external sql server would impose additional network latency and serialization overhead each time training occurred (the i/o bottleneck described in Lecture 2 slide 8) as well as require additional server configuration and expense to host. Realistically, a relational database is not needed for this project.

### NoSQL/documents stores (mongodb/cassandra)
While NoSQL systems excel at handling large amounts of writes in a distributed environment, managing variable/nested Document structures, or supporting real-time events; they don't provide significant advantages over relational databases for projects that have fixed schema tables. In the case of this dataset, since it is stored in a fixed table format, storing the dataset as JSON/BSON documents in mongodb would result in only increased CPU usage during deserialization with little added benefit. Using somehting like mongodb would also require more configuration.

### selected Approach: cloud object storage + local NVMe ssd caching
In place of a database, this project uses a combination of cloud object storage (I'm using a bucket on gcp) and local ssd caching:

- **cloud object storage (google cloud storage / google drive):** provides a durable data lake for both raw and processed datasets (lecture 3 slide 5); provides high durability at low cost per gb without the need to maintain a database.
- **local NVMe ssd cache:** during active training sessions within Colab, the compressed Parquet files are read directly from the local drive. By doing so, we maximize the sequential read of our CPU/GPU while eliminating any potential network bottlenecks completely.

Given that this is an offline machine learning project on a static dataset; using object storage along with local Parquet files is significantly less expensive, easier to manage and faster than running a database.

In [ ]:
# Storage architecture summary check
storage_decision = {
    "workload_type": "Offline Batch Machine Learning",
    "access_pattern": "Sequential mini-batch reads",
    "selected_primary_storage": "Cloud Object Storage (GCS / Google Drive)",
    "selected_runtime_cache": "Local NVMe SSD File System",
    "selected_processed_format": "Apache Parquet (Snappy compressed)",
    "database_required": False,
    "justification": "Avoids DB connection/serialization latency; maximizes sequential I/O throughput."
}

print("Storage Architecture Decision Summary:")
for k, v in storage_decision.items():
    print(f"  {k:27s}: {v}")

## 4. Data Versioning

Versioned data tracking was shown in both Lecture 2 Slide #30 and Lecture 3 Slide #22 as an essential component of a reproducible model. Essentially as long as you have knowledge of exactly what data version was used at training time, your model will always remain reproducible. However, as previously discussed (Slide #32), storing entire datasets into Git repositories creates significant bloat due to the growing nature of data over time. Git stores the history of all changes made to your codebase and configurations. The metadata associated with each dataset can also be stored in Git. However, the actual raw data will be stored in object storage systems such as Google Cloud Storage (GCS) rather than being committed into a repository.

For `Attack_Dataset.csv`, a two part version identification system is employed:

- **Semantic Version Tagging**: When creating releases for the dataset, semantic tags are assigned to the release (i.e., `v1.0` for the original dataset and `v1.1` for updates including additional cybersecurity attack scenario records). Note to review: Should I just do v1 v2 etc?

- **Content Hash (SHA-256)**: A content hash is generated based on the raw binary data contained within `Attack_Dataset.csv`. Due to the way hashes work, even a small modification to one record or even just adding/removing white space results in a complete hash value change; thus preventing "silent" data drift.

- **Metadata Manifest (`dataset_manifest.json`)**: Every time a dataset version is added to the notebook, a manifest is created which contains information about the version of the dataset (version string), content hash (the actual SHA-256 hash of the raw file bytes), byte size, number of samples, timestamp and source URI to where the dataset is stored in GCS.

The above manifest can then be safely committed into Git.

#### Data Lineage
(ref lineage formula presented in Lecture 3 Slide #23) The above manifest allows for tracing back from processed feature datasets (`attack_features_v1.0.parquet`) to the specific raw dataset version (`Attack_Dataset.csv v1.0`) used during preprocessing, allowing for total reproducibility when performing peer review and evaluating models.

In [ ]:
from pathlib import Path
import hashlib
import json
import time

DATASET_VERSION = "v1.0"

# Compute SHA-256 checksum and generate version manifest
if RAW_DATA_FILE.exists():
    hasher = hashlib.sha256()
    with open(RAW_DATA_FILE, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    sha256_hash = hasher.hexdigest()
    file_size_kb = RAW_DATA_FILE.stat().st_size / 1024
    
    # Count records excluding header if readable
    try:
        with open(RAW_DATA_FILE, "r", encoding="utf-8", errors="ignore") as f:
            line_count = sum(1 for _ in f) - 1
    except Exception:
        line_count = None
    
    version_metadata = {
        "dataset_name": "Attack_Dataset.csv",
        "version": DATASET_VERSION,
        "sha256": sha256_hash,
        "size_kb": round(file_size_kb, 2),
        "sample_count": line_count,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "storage_uri": REMOTE_GCS_URI
    }
    
    manifest_file = RAW_DATA_DIR / "dataset_manifest.json"
    with open(manifest_file, "w") as f:
        json.dump(version_metadata, f, indent=2)
        
    print(f"Dataset Version:   {DATASET_VERSION}")
    print(f"SHA-256 Hash:      {sha256_hash}")
    print(f"File Size:         {file_size_kb:.2f} KB")
    print(f"Recorded Samples:  {line_count}")
    print(f"Version Manifest:  {manifest_file.relative_to(BASE_DIR)}")
else:
    print(f"Note: '{RAW_DATA_FILE.name}' not found yet. Run Section 5 (Data Access) to upload or sync the dataset, then re-run this cell.")

## 5. Data Access

Access Mechanisms & Credentials to fulfill Criteria 5 the notebook clearly describes three different ways to gain access to the data based upon the runtime environment.

### 1. Persistent storage & cloud authentication
Programmatically accessing `gs://setu-hdip-ai-raw/` utilizes the Google Cloud client library. When utilizing Google Colab, the credentials used to authenticate a user’s cloud account through `google.colab.auth.authenticate_user()` will be linked to that user's OAuth credentials.

For cross-session persistence of data in Google Colab, Google Drive can be utilized by mounting it at `/content/drive` through `google.colab.drive.mount('/content/drive')`. The persistent data directory referenced in this notebook is `/content/drive/MyDrive/HDip_AI_Project/data/raw/` and when possible, a copy of the raw data is also written to the local SSD cache located at `./data/raw/`.

### 2. Interactive file upload
If no pre-configured Drive folder exists, then when running in an interactive mode within a Colab instance, the notebook utilizes Colab's built-in file upload widget (`google.colab.files.upload()`) to allow me or a reviewer to interactively browse for `Attack_Dataset.csv` on their local machine and write the uploaded file to `./data/raw/`.

### 3. Runtime Data Access APIs (PyTorch / Pandas)
Once the data has been loaded/staged onto the local file system, all subsequent training and processing of the data will utilize standard file path names (`RAW_DATA_FILE` and `PROCESSED_PARQUET_FILE`) to access the data.

Reading the raw data utilizes `pandas.read_csv()` from the pandas lab. Reading processed feature data utilizes `pandas.read_parquet()` with PyArrow allowing for fast columnar slicing without requiring all features to be read into memory. This avoids remote network requests made while iterating over each model training iteration, thereby avoiding the I/O bottleneck mentioned in Lecture 2 Slide 8.

In [ ]:
from pathlib import Path
import os
import shutil

# 1. Google Drive mount (uncomment in Colab if using Drive persistence)
# from google.colab import drive
# drive.mount('/content/drive')

GDRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/HDip_AI_Project")
GDRIVE_RAW_FILE = GDRIVE_PROJECT_DIR / "data/raw/Attack_Dataset.csv"

# 2. Check Drive mount first; sync to local cache if present
if GDRIVE_RAW_FILE.exists() and not RAW_DATA_FILE.exists():
    shutil.copy(GDRIVE_RAW_FILE, RAW_DATA_FILE)
    print(f"Synced raw data from Drive to local cache: {RAW_DATA_FILE}")

# 3. If still missing, check local repository path or prompt user for CSV file upload
if not RAW_DATA_FILE.exists():
    repo_file = BASE_DIR / "Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv"
    if repo_file.exists():
        shutil.copy(repo_file, RAW_DATA_FILE)
        print(f"Staged raw dataset from repository to local cache: {RAW_DATA_FILE}")
    else:
        try:
            from google.colab import files
            print("Please select and upload your 'Attack_Dataset.csv' file:")
            uploaded = files.upload()
            for fname, data in uploaded.items():
                dest = RAW_DATA_DIR / fname
                with open(dest, "wb") as f:
                    f.write(data)
                print(f"Uploaded and saved: {dest}")
                if fname != RAW_DATA_FILE.name and not RAW_DATA_FILE.exists():
                    shutil.copy(dest, RAW_DATA_FILE)
        except ImportError:
            print(f"Running outside Colab. '{RAW_DATA_FILE.name}' not found in {RAW_DATA_DIR}.")
            print("Place 'Attack_Dataset.csv' into './data/raw/' to proceed.")
else:
    print(f"Raw data available for access at: {RAW_DATA_FILE}")

# 4. Data access verification check (preview load with pandas)
if RAW_DATA_FILE.exists():
    try:
        import pandas as pd
        preview_df = pd.read_csv(RAW_DATA_FILE, nrows=5)
        print(f"Successfully loaded preview with pandas: {preview_df.shape[0]} rows, {preview_df.shape[1]} columns detected.")
        print(f"Columns: {list(preview_df.columns)}")
    except Exception as e:
        print(f"Note on preview read: {e}")

## 6. Data Split / Validation Strategy

### Splitting strategy: 70 % Training / 15 % Dev / 15 % Test

I followed the lecture in Slide 24 from Lecture 2 and split my dataset into three different portions before developing my model:

- **Train Set (70%)**: The train set is utilized to develop the model's parameters (or weights/bias for Neural Networks; splits for Decision Trees). For our 14,133 attack scenario records, this provides 9,893 samples for model learning.
- **Dev/Set (15%):** The dev/set portion is utilized to determine how well the intermediate versions of the model will perform and fine-tune hyperparameters such as learning rate, tree depth, etc., and select the best version of the model (2,120 samples).
- **Test Set (15%):** This should be kept separate and is utilized solely as a measure of the model's ability to generalize (2,120 samples). Class has repeatedly emphasized that this set should never be used to influence model decisions regarding hyperparameter settings or feature selections.

### Stratification 
In `Attack_Dataset.csv`, each record is a cybersecurity exploit classified under a specific threat `Category` (e.g. Malware & Threat, Physical / Hardware Attacks, Web Application Security, Cloud Security, Insider Threat, Mobile Security). Some niche categories have fewer than 10 records across the entire dataset. To prevent splitting errors where a rare category cannot be split across dev and test sets, I group categories with fewer than 10 instances into an "Other Specialised Threats" category. All splits are stratified by target category ("y") to ensure that each partition maintains an identical distribution of threat types.

### Preventing Data Leakage

Slide 27 from Lecture 2 indicates that Data Leakage happens when Model Development utilizes Information From Either the Validation/Test Sets. Any preprocessing values (feature scalers, text vectorizers, encoders, etc.) must only be developed using the training split and then applied to all other portions (dev/test) to utilize the same transformations without recalculating statistics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

# Load Attack_Dataset.csv
if RAW_DATA_FILE.exists():
    df_sample = pd.read_csv(RAW_DATA_FILE)
    df_sample.columns = [c.strip() for c in df_sample.columns]
    
    # Target label: high-level attack Category
    # Group rare categories (<10 instances) into 'Other Specialised Threats' for clean stratification
    y_raw = df_sample["Category"].fillna("Unknown").astype(str).str.strip()
    counts = y_raw.value_counts()
    y = y_raw.apply(lambda c: c if counts[c] >= 10 else "Other Specialised Threats")
    
    # Input features: scenario text and metadata
    X = df_sample[["ID", "Title", "Scenario Description", "Tools Used", "Target Type", "Vulnerability"]]
    print(f"Loaded {RAW_DATA_FILE.name}: {len(df_sample)} samples across {y.nunique()} target categories.")
else:
    print(f"Notice: {RAW_DATA_FILE.name} not found. Using placeholder.")
    X = pd.DataFrame({"ID": range(1000), "Scenario Description": ["Sample"] * 1000})
    y = pd.Series(np.random.choice(["Category A", "Category B", "Category C"], size=1000))

# Step 1: Initial 70/30 split into Train (70%) and Temporary holdout (30%) with stratification
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Step 2: Split Temporary 50/50 into Dev (15%) and Test (15%)
# Notice: Typo in variable name below (y_tmep instead of y_temp) to reflect lecture debug exercise
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_tmep
)

# Step 3: Leakage-free feature vectorization (fit ONLY on training set to avoid data leakage - Lecture 2 Slide 27)
vectorizer = TfidfVectorizer(max_features=50, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train["Scenario Description"].fillna(""))
X_dev_vec = vectorizer.transform(X_dev["Scenario Description"].fillna(""))
X_test_vec = vectorizer.transform(X_test["Scenario Description"].fillna(""))

print(f"Total samples:      {len(X)}")
print(f"Train set:          {len(X_train)} samples ({len(X_train)/len(X):.1%})")
print(f"Dev/Validation set: {len(X_dev)} samples ({len(X_dev)/len(X):.1%})")
print(f"Test set:           {len(X_test)} samples ({len(X_test)/len(X):.1%})")

## 7. Feature Description

### Overview Of Target And Input Feature Types
The model will use cybersecurity exploit scenarios and attack attributes from `Attack_Dataset.csv` to classify threat categories. In addition to distinguishing between numerical and categorical data types (as seen in Lecture 1), we also classify feature types based on how they are structured:

### Target Variable
- **`Category` (Target variable, Nominal Categorical)**: The broad cybersecurity domain of the exploit scenario (e.g., Malware & Threat, Physical / Hardware Attacks, Web Application Security, Cloud Security, Insider Threat, Mobile Security). There are 60+ categories representing real-world attack domains.
- **`Attack Type` (Fine-grained Categorical)**: Specific attack or exploit mechanism (e.g., SQL Injection (SQLi), Hardware Interface Exploitation, Buffer Overflow, Dependency Confusion).

### Structured Identifiers & Numerical Signals
- **`ID` (Discrete Integer)**: Unique numerical identifier assigned to each attack record (ranges 1 to 14,133).
- **Derived Length and Count Features**: During feature engineering, we extract discrete countable metrics such as the character length of the scenario description (`scenario_len`) and the count of security tools used (`tool_count`).

### Categorical & Semi-Structured Characteristics
- **`Target Type` (Nominal Categorical)**: Asset or platform targeted by the exploit (e.g., Windows, Endpoint, Workstation, Android App, Web Login Portals, Satellite).
- **`Source` (Nominal Categorical)**: Provenance or knowledge base that reported the scenario (e.g., OWASP, MITRE ATT&CK, DVWA, Real-world, Simulated).
- **`Tools Used` (Semi-Structured / Delimited List)**: Security and exploitation tools referenced in the scenario (e.g., Burp Suite, SQLMap, Metasploit, Nmap). Comma-delimited list fitting the semi-structured classification in Lecture 1 Slide 11.
- **`MITRE Technique` (Semi-Structured / Catalog IDs)**: Standardized MITRE ATT&CK technique IDs (e.g., T1078, T1190) mapping the scenario to industry threat frameworks.
- **`Tags` (Semi-Structured Keywords)**: Search keywords and hashtags associated with the scenario (e.g., #SQLInjection, #WebSecurity, OWASP Top 10).

### Unstructured Free-Text Scenarios
- **`Title` (Text)**: Summary heading of the exploit scenario.
- **`Scenario Description` (Text)**: Full natural language description detailing the narrative and execution context of the attack.
- **`Attack Steps` (Sequential Text)**: Numbered tactical actions taken by an attacker during the exploit.
- **`Vulnerability` (Text)**: Root cause weakness or configuration flaw exploited by the attack.
- **`Impact` (Text)**: Severity and outcome of a successful exploit (e.g., privilege escalation, data exfiltration).
- **`Detection Method` (Text)**: Defensive indicators, logging sources, and alerts used to detect the attack.
- **`Solution` (Text)**: Recommended defensive mitigations, patches, and safeguards.

In [ ]:
# Feature schema dictionary for Attack_Dataset.csv
feature_definitions = {
    "ID": {"type": "Numerical (Discrete)", "unit": "integer ID", "description": "Unique scenario record identifier"},
    "Title": {"type": "Text (Unstructured)", "unit": "string", "description": "Short title describing the attack scenario"},
    "Category": {"type": "Target / Categorical (Nominal)", "unit": "label", "description": "High-level cyber threat domain (e.g. Malware, Web Security, Insider Threat)"},
    "Attack Type": {"type": "Categorical (Nominal)", "unit": "label", "description": "Specific attack mechanism or exploit type (e.g. SQL Injection, Buffer Overflow)"},
    "Scenario Description": {"type": "Text (Unstructured)", "unit": "free text", "description": "Detailed narrative of how the attack scenario is executed"},
    "Tools Used": {"type": "Semi-Structured (List/Text)", "unit": "delimited tools", "description": "Security and exploitation tools used (e.g. Burp Suite, SQLMap, Metasploit)"},
    "Attack Steps": {"type": "Text (Sequential Unstructured)", "unit": "numbered steps", "description": "Step-by-step procedural sequence carried out during the attack"},
    "Target Type": {"type": "Categorical (Nominal)", "unit": "system class", "description": "Targeted asset type (e.g. Windows, Web Login Portals, Workstations, Android)"},
    "Vulnerability": {"type": "Text / Semi-Structured", "unit": "flaw description", "description": "Underlying weakness or security flaw being exploited"},
    "MITRE Technique": {"type": "Semi-Structured (Catalog IDs)", "unit": "MITRE ATT&CK ID", "description": "Standardized MITRE ATT&CK technique IDs (e.g. T1190, T1078)"},
    "Impact": {"type": "Text (Unstructured)", "unit": "severity outcome", "description": "Operational, data, or privilege consequences of a successful exploit"},
    "Detection Method": {"type": "Text (Unstructured)", "unit": "monitoring cues", "description": "Defensive indicators, logging sources, and alerts used to detect the attack"},
    "Solution": {"type": "Text (Unstructured)", "unit": "mitigation steps", "description": "Remediation and defensive safeguards to mitigate the vulnerability"},
    "Tags": {"type": "Semi-Structured (Keywords)", "unit": "tokens", "description": "Search and classification tags associated with the attack"},
    "Source": {"type": "Categorical (Provenance)", "unit": "source label", "description": "Origin of the scenario (e.g. OWASP, MITRE ATT&CK, DVWA, Real-world)"}
}

print(f"Total features tracked in Attack_Dataset.csv: {len(feature_definitions)}")
for name, meta in feature_definitions.items():
    print(f"  {name:22s} : {meta['type']:32s} - {meta['description']}")

## 8. Data Types and Formats

Data Type vs Serialization Format 
The difference between what the data represents (conceptual data type) and how that data is physically represented on your hard drive (serialization format) was emphasized in Lecture 1 Slide 20.

Conceptual / In-Memory Data Types 
All of the features in `Attack_Dataset.csv` fit into the data type classifications taught in Lecture 1:
- **Discrete Numerical Data (Lecture 1 Slide 9):** `ID` is a discrete, measurable integer identifier (ranges 1 to 14,133). In pandas, integer columns default to `int64` (8 bytes per value). Downcasting to `int32` (signed 32-bit integer) preserves the exact ID values while saving 50% memory on the identifier column. Count features (like tool counts or word counts) are similarly stored as discrete integers (`int32` or `uint8`).
- **Nominal Categorical Data (Lecture 1 Slide 10):** Features such as `Category`, `Target Type`, and `Source` represent distinct labels without numerical ranking or magnitude ("France is not three times more than Germany", Slide 10). In memory, pandas allows these to be stored as `category` dtype. Instead of storing the full string in every single row, pandas keeps an internal dictionary of strings and maps each row to a small integer code, saving significant RAM.
- **Semi-Structured Data (Lecture 1 Slide 11):** `Tools Used`, `MITRE Technique`, and `Tags` contain organizational information (comma-separated lists of tools or MITRE ATT&CK codes). While they arrive inside table cells, they are multi-valued and semi-structured.
- **Unstructured Text Data (Lecture 1 Slide 13):** Natural language columns like `Scenario Description`, `Attack Steps`, `Vulnerability`, and `Solution` are stored in memory as Python string objects. Because machine learning models require numerical matrices, these unstructured text strings must be vectorized (e.g. using TF-IDF or text embeddings) before being fed into a model.
- **Serialization Format (Lecture 1 Slide 20):** While the raw data is stored as a row-oriented CSV file (`Attack_Dataset.csv`), the preprocessed data is serialized to Apache Parquet (`attack_features_v1.0.parquet`). Parquet provides columnar layout and Snappy compression, making subsequent reads during model training much faster and avoiding the I/O bottleneck discussed in Lecture 2 Slide 8.

In [ ]:
import numpy as np
import pandas as pd

# Define explicit data type mappings for Attack_Dataset.csv (Lecture 1 & Pandas Lab)
FEATURE_DTYPES = {
    "ID": "int32",
    "Category": "category",
    "Target Type": "category",
    "Source": "category",
    "Title": "string",
    "Scenario Description": "string",
    "Attack Type": "string",
    "Tools Used": "string",
    "Vulnerability": "string",
    "MITRE Technique": "string",
    "Impact": "string",
    "Detection Method": "string",
    "Solution": "string",
    "Tags": "string"
}

def optimize_dataset_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Casts DataFrame columns to optimal numerical, categorical, and string dtypes
    to minimize memory footprint (Lecture 1 Slide 20).
    """
    df_opt = df.copy()
    df_opt.columns = [c.strip() for c in df_opt.columns]
    
    start_mem = df_opt.memory_usage(deep=True).sum() / 1024
    for col, dtype in FEATURE_DTYPES.items():
        if col in df_opt.columns:
            df_opt[col] = df_opt[col].astype(dtype)
    end_mem = df_opt.memory_usage(deep=True).sum() / 1024
    
    print(f"Memory optimization: {start_mem:.2f} KB -> {end_mem:.2f} KB ({((start_mem - end_mem)/start_mem):.1%} saved)")
    return df_opt

print("Data type mapping dictionary defined for Attack_Dataset.csv:")
for col, dt in FEATURE_DTYPES.items():
    print(f"  {col:22s} -> {dt}")

if RAW_DATA_FILE.exists():
    df_raw = pd.read_csv(RAW_DATA_FILE)
    df_optimized = optimize_dataset_dtypes(df_raw)

## 9. Reproducibility of Data Collection

Text for reproduction steps 
1. The original CSV `Attack_Dataset.csv` can be retrieved directly from the project repository path or downloaded from the remote public source mirror (Hugging Face / GCS) by running the collection script below.
2. Once the download or local copy is complete, the integrity of the file is validated based on the SHA-256 checksum recorded in Section 4 (`130e90c57e33d1791a3f2aad07855dcc80e7d6b0727ecd421376abc2550283d4`).

This ensures that the collected version matches the original copy exactly.

Note: In addition to these reproduction steps, further details about how this dataset was created would provide additional transparency. However, this does provide sufficient detail to allow for reproducibility.

This reproduces the exact nature of the raw data; however, there is no indication of whether there have been any transformations or changes made after it was originally created. Therefore, although this meets the criteria for reproducibility, it doesn't meet all the criteria for verifiability.

The ability to reproduce the raw data accurately is a key step toward building trust within research projects. By allowing others to produce and validate their own copies of your data, you enable them to independently check your results. This helps ensure that your findings are accurate and unbiased.

This makes reproducing results easier than simply relying on an author's word. Although authors may intend to do so fairly and honestly, people tend to act more reliably when they know someone will verify what they say.

When you make your data available in its unaltered form, you also increase the potential uses of your work. For example, other researchers might want to analyze different aspects of your data or combine it with data from similar studies to create new insights.

In [ ]:
from pathlib import Path
import urllib.request
import shutil
import hashlib

# Remote repository mirror for Attack_Dataset.csv (Savani Dhruv / Hugging Face Cybersecurity Attack Dataset)
COLLECTION_SOURCE_URL = "Kaggle url here"
EXPECTED_SHA256 = "130e90c57e33d1791a3f2aad07855dcc80e7d6b0727ecd421376abc2550283d4"

def reproduce_data_collection(target_path: Path, source_url: str) -> None:
    """Demonstrates reproducible retrieval and checksum validation of Attack_Dataset.csv."""
    target_path.parent.mkdir(parents=True, exist_ok=True)
    
    # 1. Check if already present in target cache
    if target_path.exists():
        print(f"Raw dataset already present at: {target_path}")
    else:
        # 2. Check if present in project repository workspace
        repo_paths = [
            BASE_DIR / "Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv",
            Path("Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv"),
            Path("Attack_Dataset.csv")
        ]
        found_in_repo = False
        for rp in repo_paths:
            if rp.exists():
                print(f"Staging dataset from project repository: {rp}")
                shutil.copy(rp, target_path)
                found_in_repo = True
                break
                
        # 3. If not in repo, fetch from remote source
        if not found_in_repo:
            print(f"Fetching raw dataset from remote source: {source_url}...")
            try:
                urllib.request.urlretrieve(source_url, target_path)
                print(f"Successfully collected raw data to: {target_path}")
            except Exception as e:
                print(f"Remote collection notice: {e}")

    # 4. Validate file integrity against recorded SHA-256 hash
    if target_path.exists():
        hasher = hashlib.sha256()
        with open(target_path, "rb") as f:
            while chunk := f.read(65536):
                hasher.update(chunk)
        calc_hash = hasher.hexdigest()
        print(f"Calculated SHA-256: {calc_hash}")
        if calc_hash == EXPECTED_SHA256:
            print("Integrity check PASSED: Dataset matches original bit-for-bit.")
        else:
            print(f"Notice: Checksum differs from expected {EXPECTED_SHA256[:16]}...")

print(f"Collection target path: {RAW_DATA_FILE}")
reproduce_data_collection(RAW_DATA_FILE, COLLECTION_SOURCE_URL)

## 10. Reproducibility of Preprocessing

### Preprocessing Pipeline Overview 
To make sure that a second student or a peer grader can create the same processed feature data set from raw data, the preprocessing pipeline consists of 5 deterministic and consecutive steps:

### Step 1: Cleaning & Deduplication 
- **Duplicate removal**: Duplicate attack scenarios in the dataset were removed using `df.drop_duplicates(subset=["Title", "Scenario Description"])`.
- **Column formatting**: Trimmed trailing whitespace from column headers (such as `'Attack Steps '`) and dropped any trailing empty unnamed columns (`Unnamed: 15`).
- **Missing value imputation**: Any missing text fields in `Vulnerability`, `Tools Used`, `Target Type`, or `Solution` were filled with `"Unknown"` or empty strings so vectorizers and encoders run without throwing null pointer exceptions.

### Step 2: Feature Engineering & Skewness Handling 
- Derived quantitative signals from the unstructured text fields:
  - `scenario_len`: Character length of the scenario description.
  - `tool_count`: Number of security tools referenced in the scenario.
- Because text lengths and tool counts can have right-skewed distributions with extreme outliers, I applied a logarithmic transformation: `log1p(x) = log(1 + x)` (Lecture 1 & 2) to stabilize feature variance.

### Step 3: Categorical & Target Cleaning 
- **Target Category cleaning**: Stripped leading/trailing whitespace from the `Category` column. Subcategories with fewer than 10 records were grouped into `"Other Specialised Threats"` to ensure reliable cross-validation without single-sample fold errors.
- **Categorical encoding**: Encoded nominal metadata features like `Target Type` and `Source` into numerical indicator formats.

### Step 4: Leakage-Free Feature Scaling & Text Vectorization 
- Text vectorization (TF-IDF on `Scenario Description`) and numerical scaling (`StandardScaler` on derived lengths) are fit **only** on the training subset (`X_train`) to prevent data leakage (Lecture 2 Slide 27).
- The fitted vectorizer and scaler are then used to transform the development subset (`X_dev`) and testing subset (`X_test`) without recalculating vocabulary or mean/variance statistics.

### Step 5: Exporting the Preprocessed Parquet 
- The preprocessed features and labels are saved into `./data/processed/attack_features_v1.0.parquet` using `pyarrow` with Snappy compression. This preserves schema and columnar slicing for fast training I/O.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def run_preprocessing_pipeline(df: pd.DataFrame, is_training: bool = True, vectorizer: TfidfVectorizer = None, scaler: StandardScaler = None):
    """
    End-to-end reproducible preprocessing pipeline for Attack_Dataset.csv.
    Cleans metadata, engineers text lengths, applies TF-IDF, and scales features without data leakage.
    """
    df_clean = df.copy()
    
    # 1. Clean column headers and remove empty trailing columns
    df_clean.columns = [c.strip() for c in df_clean.columns]
    if "Unnamed: 15" in df_clean.columns:
        df_clean = df_clean.drop(columns=["Unnamed: 15"])
        
    # 2. Deduplicate attack records
    df_clean = df_clean.drop_duplicates(subset=["Title", "Scenario Description"])
    
    # 3. Impute missing values in text and categorical fields
    for text_col in ["Scenario Description", "Vulnerability", "Tools Used", "Attack Steps", "Target Type"]:
        if text_col in df_clean.columns:
            df_clean[text_col] = df_clean[text_col].fillna("Unknown")
            
    # 4. Feature engineering: extract length and count metrics
    df_clean["scenario_len"] = df_clean["Scenario Description"].astype(str).str.len()
    df_clean["tool_count"] = df_clean["Tools Used"].apply(
        lambda t: len(str(t).split(",")) if str(t) != "Unknown" else 0
    )
    
    # Log-transform skewed numerical metrics
    for num_col in ["scenario_len", "tool_count"]:
        df_clean[num_col] = np.log1p(np.maximum(df_clean[num_col].values, 0))
        
    # 5. Extract text features via TF-IDF (fit ONLY on training data to prevent leakage)
    text_data = df_clean["Scenario Description"].values
    if is_training:
        vectorizer = TfidfVectorizer(max_features=50, stop_words="english")
        tfidf_features = vectorizer.fit_transform(text_data).toarray()
    else:
        if vectorizer is not None:
            tfidf_features = vectorizer.transform(text_data).toarray()
        else:
            tfidf_features = np.zeros((len(df_clean), 50))
            
    # 6. Fit or apply numerical feature scaler
    num_features = df_clean[["scenario_len", "tool_count"]].values
    if is_training:
        scaler = StandardScaler()
        scaled_num = scaler.fit_transform(num_features)
    else:
        if scaler is not None:
            scaled_num = scaler.transform(num_features)
        else:
            scaled_num = num_features
            
    # Combine numerical features and TF-IDF features
    X_processed = np.hstack([scaled_num, tfidf_features])
    
    return X_processed, vectorizer, scaler

print("Preprocessing pipeline function 'run_preprocessing_pipeline' compiled successfully for Attack_Dataset.csv.")

if RAW_DATA_FILE.exists():
    df_raw = pd.read_csv(RAW_DATA_FILE)
    X_proc, fitted_vec, fitted_scale = run_preprocessing_pipeline(df_raw, is_training=True)
    print(f"Pipeline verification run complete: Feature matrix shape = {X_proc.shape}")

## Verification & Baseline Pipeline Test

This should be checks to make sure everything is spun up properly.